# 00 — Shared loader and contracts

**Pregunta:** ¿estamos cargando los artifacts correctos y entendiendo sus contratos?

Este notebook no es un reporte financiero. Es el contrato de lectura para el paquete de reportes contables/profesionales.

Responsabilidades:

- localizar el repo y el bundle `public/accounting/latest`;
- verificar artifacts requeridos y opcionales;
- normalizar métricas anuales;
- listar años, monedas, `metric_id`, dimensiones y fuentes;
- detectar problemas de lectura antes de que los reportes narrativos arranquen;
- exportar outputs externos limpios a `out/professional_pack/latest`.

No responsabilidades:

- no recalcula lógica core;
- no clasifica transacciones;
- no decide saldos;
- no inventa caja;
- no suma ARS + USD;
- no cuenta todavía la historia financiera.


In [4]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "Makefile").exists() and (p / "accounting").is_dir():
        repo_root = p
        break
if repo_root is None:
    raise FileNotFoundError("Could not find repo root. Run from inside accounting-backend.")

reports_dir = repo_root / "accounting" / "notebooks" / "accounting_reports"
if str(reports_dir) not in sys.path:
    sys.path.insert(0, str(reports_dir))

from _shared import (
    find_repo_root, professional_pack_dir, inspect_artifacts, load_annual_dashboard_metrics,
    metric_inventory, dimension_inventory, qa_findings, build_readiness_summary,
    export_shared_contract_outputs, DEFAULT_REQUIRED_ARTIFACTS, DEFAULT_OPTIONAL_ARTIFACTS,
)

repo_root = find_repo_root(repo_root)
pack_dir = professional_pack_dir(repo_root)
print("repo_root:", repo_root)
print("professional_pack:", pack_dir)


repo_root: /home/matias/repos/accounting-backend
professional_pack: /home/matias/repos/accounting-backend/out/professional_pack/latest


## 1. Artifact registry

Los artifacts **required** son necesarios para que los demás reportes funcionen. Los **optional** agregan QA, drilldown o contexto, pero no deberían romper el 00 si faltan.

In [5]:
artifact_inventory = inspect_artifacts(
    repo_root,
    required=DEFAULT_REQUIRED_ARTIFACTS,
    optional=DEFAULT_OPTIONAL_ARTIFACTS,
)

display(artifact_inventory)

missing_required = artifact_inventory[
    artifact_inventory["required"].eq(True) & ~artifact_inventory["exists"].eq(True)
]

# if not missing_required.empty:
#     display(missing_required)
#     raise FileNotFoundError("Missing required artifacts. Run make run-accounting / publish before report notebooks.")


,artifact_key,required,expected_path,exists,rows,cols,columns,error
0,annual_dashboard_metrics,True,public/accounting/latest/canonical_dashboard/a...,True,287.0,24.0,"metric_id, period_grain, period, period_start,...",
1,annual_dashboard_qa,True,public/accounting/latest/canonical_dashboard/a...,True,12.0,4.0,"check, status, detail, severity",
2,metric_contract_frontier,True,public/accounting/latest/public_contract/metri...,True,22.0,18.0,"metric_id, label, semantic_category, flow_or_s...",
3,public_manifest,True,public/accounting/latest/manifest.csv,False,NaN,NaN,,
4,artifact_contracts_public,False,public/accounting/latest/public_contract/artif...,True,40.0,10.0,"name, relpath, artifact_role, accounting_natur...",
5,publish_contract_qa,False,public/accounting/latest/qa/publish_contract_q...,False,NaN,NaN,,
6,release_checks,False,out/professional_pack/latest/qa/release_checks...,False,NaN,NaN,,
7,validation_report,False,out/metrics/latest/validation_report.csv,True,2.0,4.0,"level, check_name, message, n_rows",
8,monthly_operating_statement,False,out/run/accounting/latest/monthly_operating_st...,True,1577.0,15.0,"period, period_end, Currency, statement_line, ...",
9,monthly_flow_semantic_split,False,out/run/accounting/latest/monthly_flow_semanti...,True,780.0,25.0,"period, period_end, Currency, Box, Lugar, acto...",


## 2. Load and normalize annual metrics

Fuente principal:

```text
public/accounting/latest/canonical_dashboard/annual_balance_dashboard_metrics.csv
```

La normalización acá es de lectura/reporting: `period` como año string, `Currency` explícita, `value` numérico, y columnas opcionales presentes.

In [6]:
metrics = load_annual_dashboard_metrics(repo_root)

years = sorted(metrics.loc[metrics["period"].astype(str).str.match(r"^\d{4}$", na=False), "period"].unique().tolist())
currencies = sorted(metrics["Currency"].fillna("N/A").astype(str).unique().tolist())

print("metric rows:", len(metrics))
print("years:", years)
print("currencies:", currencies)
print("metric_ids:", metrics["metric_id"].nunique())

display(metrics.head(30))


metric rows: 287
years: ['2022', '2023', '2024', '2025', '2026']
currencies: ['ARS', 'N/A', 'USD']
metric_ids: 37


,metric_id,period_grain,period,period_start,period_end,Currency,value,value_status,flow_or_stock,accounting_section,dashboard_section,dimension_name,dimension_value,source_table,source_filter,calculation_rule,frontend_suitability,public_flag,internal_flag,legacy_flag,validation_status,caveat,run_id,as_of_date,format_hint
0,IS.REVENUE.OPERATING,Y,2022,2022-01-01,2022-12-31,ARS,4948804.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
1,IS.REVENUE.OPERATING,Y,2023,2023-01-01,2023-12-31,ARS,9141302.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
2,IS.REVENUE.OPERATING,Y,2023,2023-01-01,2023-12-31,USD,0.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
3,IS.REVENUE.OPERATING,Y,2024,2024-01-01,2024-12-31,ARS,13558336.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
4,IS.REVENUE.OPERATING,Y,2024,2024-01-01,2024-12-31,USD,3610.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
5,IS.REVENUE.OPERATING,Y,2025,2025-01-01,2025-12-31,ARS,40125422.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
6,IS.REVENUE.OPERATING,Y,2025,2025-01-01,2025-12-31,USD,4560.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
7,IS.REVENUE.OPERATING,Y,2026,2026-01-01,2026-12-31,ARS,29589000.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
8,IS.REVENUE.OPERATING,Y,2026,2026-01-01,2026-12-31,USD,2280.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=operating_revenue,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,
9,IS.OPEX.PROPERTY,Y,2022,2022-01-01,2022-12-31,ARS,587266.00,available,flow,income_statement,1. Operating result,,,monthly_operating_statement.csv,statement_line=property_opex_true,annual flow = sum monthly flow by year and cur...,safe_with_caveat,True,False,False,ok,,20260701T190254Z,2026-07-01,


## 3. Readiness summary

Resumen corto para saber si el resto del pack puede correr y con qué salvedades.

In [7]:
metric_inv = metric_inventory(metrics)
dimension_inv = dimension_inventory(metrics)
qa = qa_findings(metrics, artifact_inventory)
readiness = build_readiness_summary(repo_root, metrics, artifact_inventory, qa)

display(readiness)
display(qa)


,field,value
0,repo_root,/home/matias/repos/accounting-backend
1,years_available,"2022, 2023, 2024, 2025, 2026"
2,currencies_available,"ARS, N/A, USD"
3,metric_rows,287
4,metric_ids,37
5,required_artifacts_missing,1
6,qa_failures,1
7,qa_warnings,1
8,run_latest_target,20260701T190254Z
9,metrics_latest_target,20260701T190254Z


,severity,area,check,detail
0,fail,artifacts,required_artifact_exists,public_manifest missing at public/accounting/l...
1,ok,metrics,available_metric_has_value,No available metrics with NaN value
2,ok,currency,no_cross_currency_totals,No suspicious cross-currency Currency labels f...
3,ok,cash,cash_frontend_safe_available,5 cash rows have available values
4,warning,display,hidden_metric_id_collisions,50 display key combinations map to more than o...


## 4. Metric inventory

Este inventario es el mapa de qué puede consumir cada notebook. No debe ser interpretado como reporte final; es un índice técnico/humano de métricas disponibles.

In [8]:
display(metric_inv)

operating_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith("IS.")]
funding_distribution_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith(("FUND.", "DIST.", "COV."))]
cash_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith(("BS.CASH", "DQ.CASH"))]
debt_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith("ID.DEBT")]
dq_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith("DQ.")]

print("operating metrics:", len(operating_metrics))
print("funding/distribution/coverage metrics:", len(funding_distribution_metrics))
print("cash metrics:", len(cash_metrics))
print("debt metrics:", len(debt_metrics))
print("DQ metrics:", len(dq_metrics))


,metric_id,dashboard_section,Currency,dimension_name,dimension_value,n_rows,n_values,min_value,max_value,periods,value_status,source_table,caveat,metric_label
0,BS.CASH.CLOSE.BOX,3. Cash and liquidity,N/A,,,1,0,NaN,NaN,nan,unavailable,monthly_cash_close.csv,No frontend-safe cash rows exist; no fallback ...,Caja validada por box
1,BS.CASH.FB,7. Legacy reconciliation,N/A,,,1,0,NaN,NaN,nan,not_applicable,metric_contract_frontier.csv,Legacy/reconciliation only unless rebuilt from...,BS.CASH.FB
2,BS.CASH.PM,7. Legacy reconciliation,N/A,,,1,0,NaN,NaN,nan,not_applicable,metric_contract_frontier.csv,Legacy/reconciliation only unless rebuilt from...,BS.CASH.PM
3,BS.CASH.TOTAL,3. Cash and liquidity,N/A,,,1,0,NaN,NaN,nan,unavailable,monthly_cash_close.csv,No frontend-safe cash rows exist; no fallback ...,Caja validada total
4,COV.NET.AFTER_DRAWS,2. Funding and distributions,ARS,,,5,5,-6.434288e+06,0.00,"2022, 2023, 2024, 2025, 2026",available,monthly_operating_statement.csv,,Cobertura después de funding y retiros
5,COV.NET.AFTER_DRAWS,2. Funding and distributions,USD,,,4,4,0.000000e+00,4560.00,"2023, 2024, 2025, 2026",available,monthly_operating_statement.csv,,Cobertura después de funding y retiros
6,COV.SAVINGS_RATE,2. Funding and distributions,ARS,,,5,5,-1.171567e+00,0.00,"2022, 2023, 2024, 2025, 2026",available,monthly_operating_statement.csv,Ratio of annual aggregates; not an average of ...,Savings / coverage rate
7,COV.SAVINGS_RATE,2. Funding and distributions,USD,,,4,3,9.166667e-01,1.00,"2023, 2024, 2025, 2026","available, not_applicable",monthly_operating_statement.csv,Ratio of annual aggregates; not an average of ...,Savings / coverage rate
8,DIST.DIVIDENDS,2. Funding and distributions,ARS,,,5,5,0.000000e+00,682384.00,"2022, 2023, 2024, 2025, 2026",available,monthly_operating_statement.csv,,Dividendos
9,DIST.DIVIDENDS,2. Funding and distributions,USD,,,4,4,0.000000e+00,190.00,"2023, 2024, 2025, 2026",available,monthly_operating_statement.csv,,Dividendos


operating metrics: 22
funding/distribution/coverage metrics: 15
cash metrics: 5
debt metrics: 34
DQ metrics: 7


## 5. Dimension inventory

Este inventario ayuda a detectar cómo vienen las métricas dimensionadas: por propiedad, categoría semántica, actor, debtor/creditor, o `cash_path` cuando esté disponible.

Regla de reporting: si una dimensión es importante para interpretación humana, no debe quedar escondida solo dentro del `line` label; los notebooks posteriores pueden promoverla a columna.

In [9]:
display(dimension_inv)


,dimension_name,dimension_value,n_metric_ids,metric_ids,n_rows,currencies,periods
0,,,27,"BS.CASH.CLOSE.BOX, BS.CASH.FB, BS.CASH.PM, BS....",133,"ARS, N/A, USD","2022, 2023, 2024, 2025, 2026, nan"
1,Lugar,CABA,1,IS.RENT.BY_PROPERTY,7,"ARS, USD","2022, 2023, 2024, 2025, 2026"
2,Lugar,Tigre 01,1,IS.RENT.BY_PROPERTY,5,ARS,"2022, 2023, 2024, 2025, 2026"
3,Lugar,Tigre 28,1,IS.RENT.BY_PROPERTY,5,ARS,"2022, 2023, 2024, 2025, 2026"
4,Lugar,Tigre 32,1,IS.RENT.BY_PROPERTY,5,ARS,"2022, 2023, 2024, 2025, 2026"
5,actor,Household,1,FUND.CONTRIB.BY_ACTOR,2,ARS,"2024, 2025"
6,actor,Property Management,1,FUND.CONTRIB.BY_ACTOR,2,ARS,"2024, 2025"
7,debtor_creditor,Alejandro -> MI,6,"ID.DEBT.ACTIVITY.ADJUSTMENTS, ID.DEBT.ACTIVITY...",18,USD,"2024, 2025, 2026"
8,debtor_creditor,Alejandro -> PM,6,"ID.DEBT.ACTIVITY.ADJUSTMENTS, ID.DEBT.ACTIVITY...",24,USD,"2023, 2024, 2025, 2026"
9,debtor_creditor,Hector -> MI,6,"ID.DEBT.ACTIVITY.ADJUSTMENTS, ID.DEBT.ACTIVITY...",12,USD,"2025, 2026"


## 6. QA checks del shared loader

Estos checks no reemplazan `scripts/check_release.py` ni QA del backend. Son checks de lectura para evitar reportes humanos engañosos.

In [10]:
display(qa)

available_nan = metrics[
    metrics["value_status"].astype(str).str.lower().eq("available")
    & metrics["value"].isna()
]
if not available_nan.empty:
    print("Rows with value_status=available but NaN value:")
    display(available_nan.head(100))

suspicious_currency = metrics[
    metrics["Currency"].astype(str).isin(["ALL", "Mixed", "ARS+USD", "MULTI", ""])
]
if not suspicious_currency.empty:
    print("Suspicious cross-currency rows:")
    display(suspicious_currency.head(100))


,severity,area,check,detail
0,fail,artifacts,required_artifact_exists,public_manifest missing at public/accounting/l...
1,ok,metrics,available_metric_has_value,No available metrics with NaN value
2,ok,currency,no_cross_currency_totals,No suspicious cross-currency Currency labels f...
3,ok,cash,cash_frontend_safe_available,5 cash rows have available values
4,warning,display,hidden_metric_id_collisions,50 display key combinations map to more than o...


## 7. Professional interpretation guardrails

| Área | Regla |
|---|---|
| Operación | Renta/OPEX/resultado operativo no incluye funding, deuda, dividendos ni gasto personal. |
| Funding | Un aporte no es ingreso operativo. |
| Distribuciones | Retiros, gasto personal y dividendos no son OPEX de propiedad. |
| Deuda | Deuda interna no es OPEX; debe tratarse como stock/flow financiero entre actores. |
| Caja | Si no existe `validated_cash_close` frontend-safe, caja real debe mostrarse como `s/d`. |
| Moneda | ARS y USD pueden convivir como filas con `Currency`, pero nunca se suman. |
| Box | `Box` puede ser órbita de gobernanza, no caja física. |
| Settlement | Pagos directos de inquilinos/actores cancelan obligaciones pero no implican caja PM/FB. |
| QA | `unavailable` debe mostrarse; no equivale a cero. |


## 8. Export shared outputs

Outputs mínimos:

```text
out/professional_pack/latest/shared_contract_inventory.csv
out/professional_pack/latest/shared_contract_summary.md
```

También exporta inventarios auxiliares a `tables/` y `qa/`.

In [11]:
exported = export_shared_contract_outputs(
    repo_root=repo_root,
    artifact_inventory=artifact_inventory,
    readiness=readiness,
    qa=qa,
    metric_inventory_df=metric_inv,
    dimension_inventory_df=dimension_inv,
)

for key, path in exported.items():
    print(f"{key}: {path.relative_to(repo_root)}")


shared_contract_inventory: out/professional_pack/latest/shared_contract_inventory.csv
shared_contract_summary: out/professional_pack/latest/shared_contract_summary.md
readiness_summary: out/professional_pack/latest/qa/shared_readiness_summary.csv
qa_findings: out/professional_pack/latest/qa/shared_loader_qa.csv
metric_inventory: out/professional_pack/latest/tables/shared_metric_inventory.csv
dimension_inventory: out/professional_pack/latest/tables/shared_dimension_inventory.csv


## 9. Next notebooks enabled by this contract

Propuesta gobernada:

```text
01_balance_dashboard_overview.ipynb
02_cash_and_liquidity.ipynb
03_income_rent_and_operations.ipynb
04_debt_open_items_and_reconciliation.ipynb
05_family_human_storypack.ipynb
```

Cada una debe responder una pregunta humana distinta y exportar outputs propios, sin recalcular lógica core.